# Research04 Masking Diffusion 1000 Hyperparameter and Threshold Optimization
 
 Research04는 Research03에서 best generative method로 선택된 `Masking Diffusion + generated anomaly 1000` 조건을 고정하고 RandomForest hyperparameter tuning과 threshold optimization을 수행한다.
 
 > Research02를 새 generation 방식으로 재실행한 뒤 Research03 결과를 갱신하고, 그 다음 이 노트북을 다시 실행한다. 결과 summary에는 Research02 generation metadata가 함께 저장된다.


In [6]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESEARCH01_RESULT_DIR = ROOT / "data" / "research01" / "results"
RESEARCH02_RESULT_DIR = ROOT / "data" / "research02" / "results"
RESEARCH03_RESULT_DIR = ROOT / "data" / "research03" / "results"
HYPERPARAM_SCRIPT = ROOT / "tools" / "research03_diffusion1000_hyperparameter_tuning.py"
THRESHOLD_SCRIPT = ROOT / "tools" / "research03_diffusion1000_threshold_optimization.py"
RESEARCH02_SUMMARY_PATH = RESEARCH02_RESULT_DIR / "research02_generation_summary.json"

with open(RESEARCH02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    research02_summary = json.load(f)

required_keys = {"masking_strategy", "masking_diffusion_denoiser", "temporal_block_ratio", "feature_group_ratio"}
missing = required_keys - set(research02_summary)
if missing:
    raise RuntimeError(f"Research02 generated data is stale. Rerun Research02 first. Missing keys: {sorted(missing)}")

print("ROOT:", ROOT)
print("hyperparameter script exists:", HYPERPARAM_SCRIPT.exists())
print("threshold script exists:", THRESHOLD_SCRIPT.exists())
display(research02_summary)

ROOT: c:\Users\gram\Desktop\Research
hyperparameter script exists: True
threshold script exists: True


{'generation_fix': 'Generation follows the paper-style comparison: unmasked GT-GAN and Diffusion sample full windows from noise, while masking-based methods use real anomaly windows with structured masked-value restoration.',
 'masking_strategy': 'Masking-based methods use temporal block masking and feature-group masking instead of element-wise random masking. The observed seed values are preserved, and only masked time/feature regions are restored by the generator.',
 'masking_diffusion_denoiser': 'Masking Diffusion uses a GRU temporal denoiser conditioned on the masked seed window and the binary mask, rather than flattening the window into an MLP denoiser.',
 'n_real': 247,
 'n_generated_per_method': 1000,
 'n_generated_shown_in_tsne': 247,
 'temporal_block_ratio': 0.25,
 'feature_group_ratio': 0.4,
 'residual_scale': 0.35,
 'quality_ranking': ['Masking GT-GAN',
  'Masking Diffusion',
  'Diffusion',
  'GT-GAN']}

## 1. 실험 실행

아래 코드는 필요한 때만 실행한다. 이미 결과 파일이 있으면 다시 돌릴 필요는 없다.

- hyperparameter tuning은 여러 RandomForest 후보를 validation F1 기준으로 평가한다.
- threshold optimization은 선택된 best model을 고정하고 threshold만 다시 탐색한다.


In [7]:
RUN_EXPERIMENTS = True

if RUN_EXPERIMENTS:
    for script in [HYPERPARAM_SCRIPT, THRESHOLD_SCRIPT]:
        result = subprocess.run(
            [sys.executable, str(script)],
            cwd=ROOT,
            text=True,
            capture_output=True,
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"failed: {script}")
else:
    print("RUN_EXPERIMENTS=False: using existing saved results")


[1/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
[2/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': {0: 1, 1: 2}}
[3/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': {0: 1, 1: 4}}
[4/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
[5/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': {0: 1, 1: 2}}
[6/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': {0: 1, 1: 4}}
[7/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}
[8/54] {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'class

## 2. Masking Diffusion 1000 hyperparameter tuning 결과

hyperparameter는 validation F1을 기준으로 선택한다. threshold는 Research03과 같은 방식으로 real validation에서 선택한다.

In [8]:
tuning_path = RESEARCH03_RESULT_DIR / "masking_diffusion1000_randomforest_hyperparameter_tuning.csv"
tuning_summary_path = RESEARCH03_RESULT_DIR / "masking_diffusion1000_randomforest_hyperparameter_tuning_summary.json"

tuning_df = pd.read_csv(tuning_path)
display(tuning_df[[
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "class_weight",
    "threshold_selected_on_real_validation",
    "validation_f1",
    "precision",
    "recall",
    "f1",
    "auprc",
    "pred_anomaly",
]].head(10).round(4))

with open(tuning_summary_path, "r", encoding="utf-8") as f:
    tuning_summary = json.load(f)

tuning_summary["best_result"]

,n_estimators,max_depth,min_samples_leaf,max_features,class_weight,threshold_selected_on_real_validation,validation_f1,precision,recall,f1,auprc,pred_anomaly
0,300,NaN,1,sqrt,"{0: 1, 1: 2}",0.83,0.8294,0.9184,0.9122,0.9153,0.9713,294
1,500,12.0,1,sqrt,"{0: 1, 1: 2}",0.79,0.8289,0.8758,0.9054,0.8904,0.9544,306
2,300,20.0,1,sqrt,"{0: 1, 1: 2}",0.82,0.8251,0.9100,0.9223,0.9161,0.9712,300
3,500,NaN,1,sqrt,"{0: 1, 1: 2}",0.82,0.8173,0.9030,0.9122,0.9076,0.9712,299
4,500,20.0,1,sqrt,"{0: 1, 1: 2}",0.82,0.8161,0.9088,0.9088,0.9088,0.9707,296
5,300,12.0,1,sqrt,"{0: 1, 1: 2}",0.79,0.8078,0.8498,0.8986,0.8736,0.9484,313
6,500,NaN,2,sqrt,"{0: 1, 1: 2}",0.87,0.8057,0.9713,0.6858,0.8040,0.9543,209
7,500,20.0,2,sqrt,"{0: 1, 1: 2}",0.87,0.8057,0.9667,0.6858,0.8024,0.9535,210
8,300,20.0,2,sqrt,"{0: 1, 1: 2}",0.81,0.8039,0.8502,0.8818,0.8657,0.9506,307
9,300,NaN,2,sqrt,"{0: 1, 1: 2}",0.85,0.8028,0.9607,0.7432,0.8381,0.9510,229


{'method': 'Masking Diffusion',
 'generated_anomaly_used': 1000,
 'threshold_selection': 'Best F1 on real validation_windows only.',
 'threshold_selected_on_real_validation': 0.83,
 'validation_precision': 0.9185185185185185,
 'validation_recall': 0.7560975609756098,
 'validation_f1': 0.8294314381270903,
 'precision': 0.9183673469387755,
 'recall': 0.9121621621621622,
 'f1': 0.9152542372881356,
 'auroc': 0.9961491338735504,
 'auprc': 0.9712915843090161,
 'pred_normal': 2787,
 'pred_anomaly': 294,
 'true_normal': 2785,
 'true_anomaly': 296,
 'n_estimators': 300,
 'max_depth': None,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'class_weight': {'0': 1, '1': 2}}

선택된 hyperparameter와 final-test 성능은 위 코드 셀에서 `masking_diffusion1000_randomforest_hyperparameter_tuning_summary.json`을 읽어 확인한다. 이전 실행의 threshold/F1/recall 값을 고정 문장으로 사용하지 않는다.


## 3. Threshold 최적화

기존 threshold는 precision은 높지만 recall이 낮을 수 있다. 그래서 threshold를 다시 탐색하고 다음 기준을 추가한다.

> validation precision이 0.80 이상인 threshold 중 validation recall이 가장 높은 threshold 선택

이 기준은 recall을 올리되 precision이 과도하게 무너지는 것을 막기 위한 안전 기준이다.


In [9]:
strategy_path = RESEARCH03_RESULT_DIR / "masking_diffusion1000_tuned_threshold_strategy_comparison.csv"
sweep_path = RESEARCH03_RESULT_DIR / "masking_diffusion1000_tuned_threshold_sweep.csv"

strategy_df = pd.read_csv(strategy_path)
display(strategy_df[[
    "strategy",
    "threshold",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "final_precision",
    "final_recall",
    "final_f1",
    "final_auprc",
    "final_pred_anomaly",
]].round(4))

sweep_df = pd.read_csv(sweep_path)
display(
    sweep_df.sort_values("final_f1", ascending=False)[[
        "threshold",
        "validation_precision",
        "validation_recall",
        "validation_f1",
        "final_precision",
        "final_recall",
        "final_f1",
        "final_pred_anomaly",
    ]].head(10).round(4)
)

,strategy,threshold,validation_precision,validation_recall,validation_f1,final_precision,final_recall,final_f1,final_auprc,final_pred_anomaly
0,best_validation_f1,0.83,0.9185,0.7561,0.8294,0.9184,0.9122,0.9153,0.9713,294
1,max_recall_with_validation_precision_at_least_...,0.79,0.8101,0.7805,0.7950,0.8348,0.9561,0.8913,0.9713,339
2,max_validation_recall,0.49,0.3504,1.0000,0.5190,0.2808,1.0000,0.4385,0.9713,1054


,threshold,validation_precision,validation_recall,validation_f1,final_precision,final_recall,final_f1,final_pred_anomaly
77,0.82,0.8873,0.7683,0.8235,0.9106,0.9291,0.9197,302
76,0.81,0.8750,0.7683,0.8182,0.8939,0.9392,0.9160,311
78,0.83,0.9185,0.7561,0.8294,0.9184,0.9122,0.9153,294
75,0.80,0.8514,0.7683,0.8077,0.8758,0.9527,0.9126,322
80,0.85,0.9302,0.7317,0.8191,0.9517,0.8649,0.9062,269
79,0.84,0.9242,0.7439,0.8243,0.9353,0.8784,0.9059,278
74,0.79,0.8101,0.7805,0.7950,0.8348,0.9561,0.8913,339
81,0.86,0.9286,0.7134,0.8069,0.9537,0.8345,0.8901,259
82,0.87,0.9431,0.7073,0.8084,0.9565,0.8176,0.8816,253
73,0.78,0.7647,0.7927,0.7784,0.7983,0.9628,0.8729,357


추천 threshold는 위 strategy table에서 선택한다. 기본 기준은 validation precision floor를 만족하는 threshold 중 validation recall이 가장 높은 값이다. 최종 precision, recall, F1은 재실행된 `masking_diffusion1000_tuned_threshold_strategy_comparison.csv` 기준으로 반영한다.


## 4. Research01 hyperparameter baseline과 비교

아래 비교는 같은 RandomForest hyperparameter를 원본 데이터 baseline에도 적용했을 때와, Research02 Masking Diffusion 생성 anomaly 1000개를 추가하고 threshold를 최적화했을 때를 비교한다.

In [10]:
research01_tuned_path = RESEARCH01_RESULT_DIR / "randomforest_hyperparameter_applied_baseline.csv"
research01_tuned_df = pd.read_csv(research01_tuned_path)
research04_selected = strategy_df.loc[
    strategy_df["strategy"].str.startswith("max_recall_with_validation_precision")
].iloc[0]
research01_selected = research01_tuned_df.iloc[0]

comparison_df = pd.DataFrame([
    {
        "experiment": "Research01 tuned baseline (no generated data)",
        "generated_anomaly_used": int(research01_selected["generated_anomaly_used"]),
        "threshold": research01_selected["threshold_selected_on_real_validation"],
        "precision": research01_selected["precision"],
        "recall": research01_selected["recall"],
        "f1": research01_selected["f1"],
        "auprc": research01_selected["auprc"],
        "pred_anomaly": int(research01_selected["pred_anomaly"]),
    },
    {
        "experiment": "Research04 Masking Diffusion 1000 tuned RF + threshold optimization",
        "generated_anomaly_used": 1000,
        "threshold": research04_selected["threshold"],
        "precision": research04_selected["final_precision"],
        "recall": research04_selected["final_recall"],
        "f1": research04_selected["final_f1"],
        "auprc": research04_selected["final_auprc"],
        "pred_anomaly": int(research04_selected["final_pred_anomaly"]),
    },
])

display(comparison_df.round(4))

delta = comparison_df.set_index("experiment").diff().iloc[-1]
display(pd.DataFrame(delta, columns=["Research04 Masking Diffusion - Research01 tuned baseline"]).round(4))

,experiment,generated_anomaly_used,threshold,precision,recall,f1,auprc,pred_anomaly
0,Research01 tuned baseline (no generated data),0,0.82,0.9435,0.9020,0.9223,0.9739,283
1,Research04 Masking Diffusion 1000 tuned RF + t...,1000,0.79,0.8348,0.9561,0.8913,0.9713,339


,Research04 Masking Diffusion - Research01 tuned baseline


## 5. 결론 작성 방향
 
 Research04 결론은 재실행된 Research02/03 결과를 바탕으로 다시 작성한다. 특히 다음 항목을 비교한다.
 
 - Research02의 새 generated windows를 사용한 Masking Diffusion 1000 + tuned RF 성능
 - precision floor 기반 threshold optimization 후 recall/F1 변화
 - 같은 RF hyperparameter를 적용한 real-only baseline과의 trade-off
 
 이전 실행에서 나온 고정 수치 대신, 노트북의 comparison table과 delta table 값을 그대로 반영한다.
